# tests/generate_fake_data — run the demo WITHOUT Synergy creds

Writes synthetic JSON for **every** entity in `synergy_schemas.ENTITIES` into the `raw_data` Volume,
exactly where `01_ingest` would land it. The generator is **schema-driven**: it reads each entity's
`SILVER_COLUMNS` and fabricates a nested row that populates every projected path (correct nesting + types),
so `02_bronze` → `03_silver` produce fully-populated tables for all entities offline — ideal for the
handoff before live credentials are provisioned.

In [ ]:
# --- dual-mode setup: runs locally via Databricks Connect AND inside a Databricks Git Folder ---
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# ensure the medallion schemas + raw Volume exist (idempotent — safe to re-run)
for sch in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{sch}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

def upload_json(volume_path: str, payload) -> int:
    """Write a JSON payload to the Volume via the Files API (network upload, no FUSE needed)."""
    body = json.dumps(payload).encode("utf-8")
    w.files.upload(volume_path, body, overwrite=True)
    return len(body)
print("volume ready:", VOLUME_PATH)

In [ ]:
import random
random.seed(7)
from synergy_schemas import ENTITIES, SILVER_COLUMNS, PRIMARY_KEYS

N = 12  # synthetic rows per entity

def dummy(typ, alias, i):
    if typ in ("int", "bigint"):
        return (100000 + i) if "mlbam" in alias else (i + 1)   # stable MLBAM space so teams/games conform
    if typ == "double":    return round(1.0 + i * 0.5, 2)
    if typ == "boolean":   return (i % 2 == 0)
    if typ == "timestamp": return "2024-04-01T18:00:00Z"
    if typ == "date":      return "2024-04-01"
    return f"{alias}-{i}"  # string

def set_path(obj, path, value):
    parts = path.split(":")
    for p in parts[:-1]:
        obj = obj.setdefault(p, {})
    obj[parts[-1]] = value

def build_rows(entity, cols, pk, n):
    rows = []
    for i in range(n):
        row = {}
        for path, alias, typ in cols:
            set_path(row, path, dummy(typ, alias, i))
        for k in pk:                       # unique, readable natural key
            set_path(row, k, f"{entity}_{i:05d}")
        rows.append(row)
    return rows

for e in ENTITIES:
    name = e["name"]
    rows = build_rows(name, SILVER_COLUMNS[name], PRIMARY_KEYS[name], N)
    upload_json(f"{VOLUME_PATH}/{name}/{name}_synthetic.json", rows)
    print(f"  {name:28} {len(rows):>3} rows x {len(SILVER_COLUMNS[name]):>3} cols")
print("\nNow run 02_bronze_autoloader then 03_silver_transformations — no creds needed.")